# 04 · Iterace učení, neúspěšné experimenty a opakování

Tyto notebooky používají skutečné soubory projektu, ne vymyšlené výsledky. Buňky bez dat skončí vysvětlením, které balíčky chybí. Klon repozitáře může vyžadovat přístup; alternativně nahraj jeho kopii do Colab. API klíč pro tuto analýzu nepotřebuješ. Trénování ani nové vyhledávání se bez přepínače nespouští. Rezervované testové otázky se nečtou ani nevyhodnocují.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib
# In Colab clone the project; locally reuse the current checkout.
if Path('src/librarian').is_dir():
    ROOT = Path.cwd()
elif Path('ML_Librarian/src/librarian').is_dir():
    ROOT = Path('ML_Librarian').resolve()
else:
    subprocess.run(['git','clone','https://github.com/Kilitar/ML_Librarian.git'],check=True)
    ROOT = Path('ML_Librarian').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src/librarian'))
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
# Set an immutable git ref before starting for exact code-version reproduction.


In [ ]:
# Optional environment setup. Enable in a fresh Colab runtime; CPU is sufficient.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-librarian.txt'],check=True)


In [ ]:
# Operator supplies the ignored data bundle(s), never an API key.
UPLOAD_BUNDLES = False  # In Colab set True and select runtime.local.zip and/or development.local.zip.
if UPLOAD_BUNDLES:
    from google.colab import files
    uploaded = files.upload()
    import io, zipfile
    expected = json.loads((ROOT/'deployment/artifacts.json').read_text())
    for filename, blob in uploaded.items():
        record = expected.get(filename)
        if record is None or hashlib.sha256(blob).hexdigest() != record['sha256']:
            raise ValueError('Unknown/stale bundle: compare with operator manifest; do not bypass SHA check')
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            names=z.namelist()
            manifest=json.loads(z.read('bundle_manifest.json'))
            if len(names)!=len(set(names)) or set(names)!=set(manifest)|{'bundle_manifest.json'}:
                raise ValueError('Invalid bundle manifest')
            if sum(i.file_size for i in z.infolist())>40*1024*1024:raise ValueError('Bundle too large')
            staged=[]
            for name, digest in manifest.items():
                if name not in record['files']:raise ValueError('Unexpected member')
                target=(ROOT/name).resolve()
                if not target.is_relative_to(ROOT.resolve()):raise ValueError('Unsafe path')
                content=z.read(name)
                if hashlib.sha256(content).hexdigest()!=digest:raise ValueError('Member SHA mismatch')
                staged.append((target,content))
            for target,content in staged:
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
        print('Verified bundle:',filename)


In [ ]:
for filename in ['supervised_pair_summary.json','supervised_pair_v2_summary.json','source_group_extension_summary.json']:
    data=json.loads((ROOT/'eval'/filename).read_text(encoding='utf-8'))
    print(filename)
    print(json.dumps(data.get('checks',data),ensure_ascii=False,indent=2))


In [ ]:
path=ROOT/'eval/supervised_pair_v2.local.json'
if path.exists():
    from train_supervised_pair_v2 import evaluate
    raw=json.loads(path.read_text(encoding='utf-8'))
    for variant in ['base','v1','v2']:
        print(variant,evaluate(raw['rows'],variant,0.5))
    for row in raw['rows']:
        print(row['query'],row['label'],row['base'],row['v1'],row['v2'])
        print(row['input'])
else:
    print('Pro úplné pasáže a přepočet skutečných výsledků nahraj development.local.zip.')


## Původní trénovací kód
Zobrazíme použitý blok z v1 a konfiguraci. Žádný experiment není označen úspěšným jen podle poklesu loss.

In [ ]:
print((ROOT/'config/supervised_pair_experiment.json').read_text())
print((ROOT/'src/librarian/train_supervised_pairs.py').read_text(encoding='utf-8'))

## Volitelné opakování posledního bloku a hlavy
Skutečná data, stejná pravidla zmrazení vrstev/optimalizace. Nový běh se ukládá odděleně a není bitově shodnou reprodukcí původního CPU/hardware. Defaultně vypnuto. Nenahradí held-out přejímku.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import random, numpy as np, torch
    from transformers import AutoTokenizer,AutoModelForSequenceClassification
    from safetensors.torch import save_file
    config=json.loads((ROOT/'config/supervised_pair_experiment.json').read_text())
    pairs=json.loads((ROOT/'eval/supervised_pair_training.local.json').read_text())['pairs']
    split=json.loads((ROOT/'eval/supervised_pair_split.json').read_text())
    train=[p for p in pairs if p['question_id'] not in split['excluded_training_question_ids'] and p['target'] is not None]
    random.seed(config['seed']);np.random.seed(config['seed']);torch.manual_seed(config['seed'])
    tokenizer=AutoTokenizer.from_pretrained(config['base_model'],revision=config['revision'],trust_remote_code=False)
    model=AutoModelForSequenceClassification.from_pretrained(config['base_model'],revision=config['revision'],use_safetensors=True,trust_remote_code=False)
    selected=[(p,tokenizer(p['query'],p['input'],truncation=False)) for p in train]
    selected=[(p,e) for p,e in selected if len(e['input_ids'])<=config['max_tokens']]
    positives=sum(p['target'] for p,e in selected);negatives=len(selected)-positives
    assert positives and negatives
    last=len(model.roberta.encoder.layer)-1
    for name,p in model.named_parameters():p.requires_grad_(name.startswith(f'roberta.encoder.layer.{last}.') or name.startswith('classifier.'))
    blocks=[p for n,p in model.named_parameters() if p.requires_grad and not n.startswith('classifier.')]
    head=[p for n,p in model.named_parameters() if p.requires_grad and n.startswith('classifier.')]
    optimizer=torch.optim.AdamW([{'params':blocks,'lr':config['learning_rate_last_block']},{'params':head,'lr':config['learning_rate_classifier']}],weight_decay=config['weight_decay'])
    criterion=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negatives/positives))
    for epoch in range(config['epochs']):
        model.train();random.shuffle(selected);losses=[]
        for start in range(0,len(selected),config['batch_size']):
            batch=selected[start:start+config['batch_size']]
            encoded=tokenizer.pad([e for p,e in batch],padding=True,return_tensors='pt')
            targets=torch.tensor([p['target'] for p,e in batch],dtype=torch.float32)
            optimizer.zero_grad();loss=criterion(model(**encoded).logits.reshape(-1),targets)
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),config['max_grad_norm']);optimizer.step()
            losses.append(float(loss.detach()))
        print('Epoch',epoch+1,'loss',sum(losses)/len(losses))
    out=ROOT/'.cache/colab_reproduction';out.mkdir(parents=True,exist_ok=True)
    save_file({n:p.detach().cpu().contiguous() for n,p in model.named_parameters() if p.requires_grad},str(out/'trainable.safetensors'))
    print('Nový experiment, žádná produkční změna ani automatické promotion:',out)
else:
    print('Trénování vypnuto. Uložené skutečné výsledky jsou dostupné výše.')
